In [2]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Microsoft YaHei'
plt.rcParams['axes.unicode_minus' ] = False

In [3]:
from lib.model import Model
from lib.dataset import Data
from lib.dataloader.generators import ChunkedGenerator, UnchunkedGenerator
import os
import importlib
import json
import torch
import numpy as np
import cv2
from lib.visualization.utils import *

/home/wen/miniconda3/envs/ray3d/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
exp_path = "/home/wen/ws/ray3d/Ray3D/checkpoint/VIDEOPOSE_h36m_videoOri_FRAME81_LR0.0005_EPOCH256_BATCH256_May_21_2025_00_07_39"
model_cfg_json_path = os.path.join(exp_path, "configs", "model_config.json")
dataset_cfg_json_path = os.path.join(exp_path, "configs", "data_config.json")
ckpt_path = os.path.join(exp_path, "epoch_8.bin")

In [21]:
model_cfg, data_cfg = {},{} 
with open(model_cfg_json_path, 'r') as f, open (dataset_cfg_json_path, 'r') as f1:
    model_cfg = json.load(f) 
    data_cfg = json.load(f1) 
receptive_field = model_cfg["NUM_FRAMES"]
pad = (receptive_field - 1) // 2  # Padding on each sid
model_cfg, data_cfg, receptive_field

({'MODEL': 'videoOri',
  'NUM_COARSE_ANG': 8,
  'TRAJECTORY_MODEL': True,
  'BONE_COMPARISON': False,
  'ARCHITECTURE': '3,3,3,3',
  'DROPOUT': 0.25,
  'NUM_FRAMES': 81,
  'CAUSAL': False,
  'CHANNELS': 1024,
  'DENSE': False,
  'NUM_KPTS': 17,
  'INPUT_DIM': 2,
  'DISABLE_OPTIMIZATIONS': False,
  'PRETRAIN': ''},
 {'DATASET': 'h36m',
  'WORLD_3D_GT_EVAL': True,
  'KEYPOINTS': 'gt',
  'TRAIN_SUBJECTS': 'S1,S5,S6,S7,S8',
  'TEST_SUBJECTS': 'S9,S11',
  'GT_3D': 'data/h36m/data_3d_h36m.npz',
  'GT_2D': 'data/h36m/data_2d_h36m_gt.npz',
  'CAMERA_PARAM': '',
  'SUBSET': 1,
  'STRIDE': 1,
  'DOWNSAMPLE': 1,
  'ACTIONS': '*',
  'REMOVE_IRRELEVANT_KPTS': False,
  'FRAME_PATH': '/ssd/yzhan/data/benchmark/3D/showroom/20210702/frame/',
  'ORI_ENCODING': True,
  'INTRINSIC_ENCODING': True,
  'RAY_ENCODING': True,
  'ADD_HEIGHT': False},
 81)

In [6]:
train_delegator = Model(model_cfg, {"ADD_HEIGHT": False}, is_train=False)
model =train_delegator.get_pos_model(); 
    

In [7]:
checkpoint = torch.load(ckpt_path, map_location=lambda storage, loc: storage)

In [8]:
model.load_state_dict(checkpoint["model_pos"], strict=True)
model.eval()

DataParallel(
  (module): TemporalModel(
    (drop): Dropout(p=0.25, inplace=False)
    (relu): ReLU(inplace=True)
    (expand_bn): BatchNorm2d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (cls_head): Sequential(
      (0): Flatten(start_dim=1, end_dim=-1)
      (1): Linear(in_features=1024, out_features=256, bias=True)
      (2): ReLU()
      (3): Linear(in_features=256, out_features=8, bias=True)
    )
    (reg_head): Sequential(
      (0): Flatten(start_dim=1, end_dim=-1)
      (1): Linear(in_features=1024, out_features=256, bias=True)
      (2): ReLU()
      (3): Linear(in_features=256, out_features=1, bias=True)
      (4): Tanh()
    )
    (expand_conv): Conv2d(34, 1024, kernel_size=(3, 1), stride=(1, 1), bias=False)
    (layers_conv): ModuleList(
      (0): Conv2d(1024, 1024, kernel_size=(3, 1), stride=(1, 1), dilation=(3, 1), bias=False)
      (1): Conv2d(1024, 1024, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (2): Conv2d(1024, 1024, kernel_

In [9]:
pose_data = Data(data_cfg)
subjects_train = data_cfg["TRAIN_SUBJECTS"].split(",")
subjects_test = data_cfg["TEST_SUBJECTS"].split(",")
action_filter = None if data_cfg["ACTIONS"] == "*" else data_cfg["ACTIONS"].split(",")
cameras_valid, poses_valid, poses_valid_2d, poses_valid_ori = pose_data.fetch_via_subject(
        subjects_test, action_filter
)
kps_left, kps_right = pose_data.get_2d_kpts()
joints_left, joints_right = pose_data.get_3d_joints()


-------------cal ori-------
-------------cal ori done-------


In [10]:

test_generator = ChunkedGenerator(
    1,
    cameras_valid,
    poses_valid,
    poses_valid_2d,
    poses_valid_ori,
    1,
    pad=pad,
    causal_shift=False,
    shuffle=False,
    augment=False,
    kps_left=kps_left,
    kps_right=kps_right,
    joints_left=joints_left,
    joints_right=joints_right,
)

In [19]:
from re import L
from matplotlib.pyplot import arrow
from tqdm import tqdm
from lib.train_val.trainer import postprocess_angle
i = 0

for cam, batch, batch_2d, batch_ori in tqdm(test_generator.next_epoch(), desc="processing"):
    kps_img= draw_kps_to_image(batch_2d[0, batch_2d.shape[1] // 2])
    
    inputs_2d = torch.from_numpy(batch_2d.astype("float32"))
    # print(inputs_2d.shape)
    out_cls, out_reg = model(inputs_2d)
    final_angles, conf = postprocess_angle(out_cls, out_reg, 8)
    ang_img = draw_orientation(None,batch_ori.squeeze(), conf)
    ang_img = draw_orientation(ang_img, final_angles.cpu().detach().numpy().squeeze(),  conf,arrow_color=(255,0,0))
    if (i % 1000 == 0):
        cv2.imwrite(f"/tmp/temp_imgs/{i}.jpg",cv2.hconcat([kps_img, ang_img]))
    i += 1
    if i > 10:
        break

processing: 10it [00:00, 90.92it/s]

tensor([[ 15.4909, -12.4003, -13.1934, -14.5046, -12.5231, -13.9506, -13.0554,
         -13.3038]], device='cuda:0', grad_fn=<AddmmBackward0>)
tensor([[ 15.4706, -12.3788, -13.1691, -14.4773, -12.5043, -13.9309, -13.0322,
         -13.2777]], device='cuda:0', grad_fn=<AddmmBackward0>)
tensor([[ 15.4488, -12.3565, -13.1450, -14.4495, -12.4860, -13.9111, -13.0092,
         -13.2518]], device='cuda:0', grad_fn=<AddmmBackward0>)
tensor([[ 15.4263, -12.3325, -13.1197, -14.4196, -12.4678, -13.8903, -12.9846,
         -13.2238]], device='cuda:0', grad_fn=<AddmmBackward0>)
tensor([[ 15.4033, -12.3063, -13.0930, -14.3890, -12.4476, -13.8675, -12.9591,
         -13.1940]], device='cuda:0', grad_fn=<AddmmBackward0>)
tensor([[ 15.3823, -12.2821, -13.0687, -14.3603, -12.4286, -13.8467, -12.9349,
         -13.1658]], device='cuda:0', grad_fn=<AddmmBackward0>)
tensor([[ 15.3581, -12.2538, -13.0398, -14.3278, -12.4063, -13.8233, -12.9064,
         -13.1340]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [ ]:
81 * 17

1377

In [ ]:
batch_2d.min(axis=(0,1,2)) , batch_2d.max(axis=(0, 1,2))

In [ ]:
from IPython.display import HTML  # 用于在Jupyter中显示动画
plt.imshow(cv2.hconcat([draw_kps_to_image(batch_2d[0, batch_2d.shape[1] // 2]), draw_orientation(None,batch_ori.squeeze())]))

In [ ]:
plt.imshow(draw_orientation(None,batch_ori.squeeze()))